# 01c · Color dominante de las fotos de Polyvore (**se ejecuta en Google Colab**)

Las imágenes pesan 3,4 GB, así que este notebook corre en Colab: descarga las fotos allá, calcula el color dominante de cada producto con `src/modelo_fly/color_imagen.py` y devuelve **solo una tabla** (`color_imagen_polyvore.csv`, unos pocos MB). No guarda imágenes en tu computadora.

**Fuente de las imágenes:** Re-PolyVore en Kaggle (`akshaypujari124/polyvore-images`), copia de las fotos del dataset de Han et al. (2017). Los archivos se llaman `<set_id>_<índice>.jpg`, igual que nuestro `item_key`.

### Antes de ejecutar (una sola vez)
1. En Colab, abre el panel de la izquierda y haz clic en la **llave 🔑 (Secretos)**.
2. Crea dos secretos copiando los valores de tu `kaggle.json` (ábrelo con el Bloc de notas):
   - `KAGGLE_USERNAME` → el valor de `"username"`
   - `KAGGLE_KEY` → el valor de `"key"`
3. Activa el interruptor **"Acceso del notebook"** en los dos.

### Ejecutar
*Entorno de ejecución → Ejecutar todo*. Tarda unos 15–25 minutos. Al final se descarga `color_imagen_polyvore.csv`: muévelo a `modelo-fly\data\processed\hito3\` en tu computadora.

In [ ]:
# Rama del repositorio con el código del color. Después del merge del PR de la etapa 3, cambiar a "main".
RAMA = "hito3/etapa3-polyvore"
REPO = "https://github.com/sowiexsker894/Predicci-n-de-compatibilidad-de-outfits-desde-el-conectoma-del-Giant-Fiber-System.git"

import os, sys, subprocess
if not os.path.exists("/content/modelo-fly"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", RAMA, REPO, "/content/modelo-fly"], check=True)
sys.path.insert(0, "/content/modelo-fly/src")
from modelo_fly.color_imagen import color_dominante
print("Código del color cargado desde la rama", RAMA)

In [ ]:
from google.colab import userdata

# El token se lee de los Secretos de Colab; no se escribe en ninguna celda.
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

DATASET = "akshaypujari124/polyvore-images"
ZIP = "/content/img/polyvore-images.zip"
if not os.path.exists(ZIP):
    subprocess.run(["kaggle", "datasets", "download", DATASET, "-p", "/content/img"], check=True)
print(f"{ZIP}: {os.path.getsize(ZIP) / 1e9:.2f} GB")

## Color de cada foto

Se lee cada imagen directamente desde el zip (sin descomprimirlo) y se procesa en paralelo.

In [ ]:
import zipfile
from concurrent.futures import ProcessPoolExecutor
from pathlib import PurePosixPath

import pandas as pd
from PIL import Image

with zipfile.ZipFile(ZIP) as z:
    nombres = [n for n in z.namelist() if n.lower().endswith((".jpg", ".jpeg", ".png"))]
print("Imágenes en el zip:", len(nombres))


def procesar(lote):
    filas = []
    with zipfile.ZipFile(ZIP) as z:
        for nombre in lote:
            ruta = PurePosixPath(nombre)
            fila = {"item_key": ruta.stem, "carpeta": ruta.parent.name}
            try:
                with z.open(nombre) as fh:
                    fila.update(color_dominante(Image.open(fh)))
                fila["error"] = ""
            except Exception as e:  # imagen dañada: se registra y se sigue
                fila["error"] = type(e).__name__
            filas.append(fila)
    return filas


TAM = 2000
lotes = [nombres[i:i + TAM] for i in range(0, len(nombres), TAM)]
filas = []
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    for i, parte in enumerate(ex.map(procesar, lotes), start=1):
        filas.extend(parte)
        if i % 10 == 0 or i == len(lotes):
            print(f"{i}/{len(lotes)} lotes · {len(filas)} imágenes")

colores = pd.DataFrame(filas)
print("Con error:", int((colores["error"] != "").sum()))
print("item_key repetidos (misma foto en dos carpetas):", int(colores["item_key"].duplicated().sum()))
colores = colores.drop_duplicates("item_key")

## Resumen y descarga

In [ ]:
display(colores["tono_img"].value_counts().to_frame("imágenes"))
display(colores["familia_img"].value_counts().to_frame("imágenes"))
display(colores[["frac_prenda", "frac_tono"]].describe().round(3))

SALIDA = "/content/color_imagen_polyvore.csv"
colores.to_csv(SALIDA, index=False, encoding="utf-8")
print(f"{SALIDA}: {len(colores)} filas, {os.path.getsize(SALIDA) / 1e6:.1f} MB")

from google.colab import files
files.download(SALIDA)